# PRE-00 — CICIDS2017 Governance, Audit, and Model-View Freeze


These notebooks implement **Sections 1–7 only** of the frozen Experiment 1 v2
pre-modeling protocol. They terminate at a machine-readable and human-readable
**PRE-PARTITION READINESS GATE**.

They do not create experimental partitions, fit learned preprocessing, train AE/DAE
models, compute reconstruction errors, or overwrite the frozen merged master.


# Scope and prohibited operations

This notebook is a **pre-partition governance/audit notebook**. Its role is to freeze
dataset identity, structural evidence, taxonomy, immutable source observation identity,
the raw model-view feature policy, and duplicate-leakage groups.

The notebook must not:

- download or reconstruct a replacement dataset;
- re-merge source files;
- overwrite the frozen master;
- delete or relabel observations;
- create training/early-stop/calibration/development/final-test partitions;
- fit imputers, scalers, encoders, clipping limits, variance thresholds, rare-category rules,
  or reconstruction thresholds;
- train any model;
- use family metadata as an AE/DAE input.

Where evidence is unavailable or ambiguous, the relevant scientific gate fails.


In [1]:
# ============================================================
# 3. EDITABLE CONFIGURATION
# ============================================================
from pathlib import Path

DATASET_NAME = "CICIDS2017"
DATASET_PREFIX = "CICIDS2017"
NOTEBOOK_STAGE = "PRE_00_SECTIONS_1_TO_7"
PROTOCOL_VERSION = "Experiment_1_v2"

# User-supplied project location.
PROJECT_ROOT = Path("/content/drive/MyDrive/EXPERIMENT1_V2")
DATASET_ROOT = PROJECT_ROOT / "DATASETS" / "CICIDS2017"

# The user supplied this MERGED folder. Leave MASTER_DATASET_PATH=None to
# auto-resolve exactly one CSV/Parquet master from it; set an explicit path if
# the folder contains more than one supported file.
MASTER_DATASET_DIR = Path(
    "/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/MERGED"
)
MASTER_DATASET_PATH = None

# Optional explicit overrides. When None, Section 2 searches the sibling AUDIT
# directory for evidence that cryptographically matches the resolved master.
AUDIT_SEARCH_DIR = DATASET_ROOT / "AUDIT"
MERGE_MANIFEST_PATH = None
SOURCE_HASH_MANIFEST_PATH = None

OUTPUT_DIR = DATASET_ROOT / "preflight_s1_s7_upstream_gate_fix_v2"
GLOBAL_DIR = PROJECT_ROOT / "global"

# Memory-conscious execution controls; these affect batching only.
CHUNK_SIZE = 50_000
ROW_ID_CHUNK_SIZE = 200_000
CARDINALITY_COLUMN_BATCH_SIZE = 4

# Descriptive-only near-constant criterion. It is never reused as learned
# feature selection.
NEAR_CONSTANT_MAX_UNIQUE = 20
NEAR_CONSTANT_DOMINANT_FRACTION = 0.999

MANIFESTS_DIR = OUTPUT_DIR / "manifests"
AUDITS_DIR = OUTPUT_DIR / "audits"
TAXONOMY_DIR = OUTPUT_DIR / "taxonomy"
IDENTITY_DIR = OUTPUT_DIR / "identity"
FEATURE_POLICY_DIR = OUTPUT_DIR / "feature_policy"
FINGERPRINTS_DIR = OUTPUT_DIR / "fingerprints"
READINESS_DIR = OUTPUT_DIR / "readiness"
SCRATCH_DIR = OUTPUT_DIR / "_scratch"


In [2]:
# ============================================================
# 4. OPTIONAL GOOGLE DRIVE MOUNT
# ============================================================
# Run this cell in Google Colab when the configured paths live in My Drive.
# It is safe to skip outside Colab or when the files are already locally mounted.

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Google Colab is not detected; Drive mount skipped.")


Mounted at /content/drive


In [3]:
# ============================================================
# 5. IMPORTS AND SOFTWARE ENVIRONMENT
# ============================================================
import csv
import gc
import hashlib
import json
import math
import os
import platform
import re
import struct
import sys
import tempfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception as exc:
    raise RuntimeError(
        "PyArrow is required for memory-conscious row-level governance manifests."
    ) from exc

EXECUTION_UTC = datetime.now(timezone.utc).isoformat()

SOFTWARE_ENVIRONMENT = {
    "python_version": sys.version,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "pyarrow_version": pa.__version__,
    "platform": platform.platform(),
    "execution_timestamp_utc": EXECUTION_UTC,
}

display(pd.DataFrame(
    [{"component": k, "value": v} for k, v in SOFTWARE_ENVIRONMENT.items()]
))


,component,value
0,python_version,"3.13.15 (main, Aug 6 2026, 11:06:22) [GCC 13...."
1,pandas_version,2.2.3
2,numpy_version,2.1.3
3,pyarrow_version,23.0.1
4,platform,Linux-6.6.122+-x86_64-with-glibc2.39
5,execution_timestamp_utc,2026-09-14T08:23:35.731576+00:00


In [4]:
# ============================================================
# 6. REUSABLE GOVERNANCE/AUDIT UTILITIES
# ============================================================

SECTION_STATUS = {str(i): "NOT_RUN" for i in range(1, 8)}
BLOCKING_FAILURES = []
NONBLOCKING_WARNINGS = []
ARTIFACT_RECORDS = []

def canonical_json_bytes(obj):
    return json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path, block_size=16 * 1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def norm_name(x):
    return "".join(ch for ch in str(x).strip().casefold() if ch.isalnum())

def resolve_unique_column(columns, requested):
    target = norm_name(requested)
    matches = [c for c in columns if norm_name(c) == target]
    return (matches[0] if len(matches) == 1 else None), matches

def record_failure(section, message):
    s = str(section)
    SECTION_STATUS[s] = "FAIL"
    BLOCKING_FAILURES.append({"section": s, "message": str(message)})
    print(f"[SECTION {s} BLOCKING FAILURE] {message}")

def record_warning(section, message):
    s = str(section)
    NONBLOCKING_WARNINGS.append({"section": s, "message": str(message)})
    print(f"[SECTION {s} WARNING] {message}")

def pass_section(section):
    s = str(section)
    if SECTION_STATUS[s] != "FAIL":
        SECTION_STATUS[s] = "PASS"

def dependencies_pass(*sections):
    return all(SECTION_STATUS[str(s)] == "PASS" for s in sections)

def markdown_table(df, max_rows=100):
    view = df.head(max_rows)
    def esc(x):
        return str(x).replace("|", r"\|").replace("\n", " ")
    lines = [
        "| " + " | ".join(esc(c) for c in view.columns) + " |",
        "| " + " | ".join("---" for _ in view.columns) + " |",
    ]
    for row in view.itertuples(index=False, name=None):
        lines.append("| " + " | ".join(esc(v) for v in row) + " |")
    if len(df) > max_rows:
        lines.append(f"\n_Only the first {max_rows} of {len(df)} rows are displayed._")
    return "\n".join(lines)

def atomic_write_bytes(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        mode="wb", delete=False, dir=str(path.parent), prefix=path.name + ".tmp."
    ) as tmp:
        tmp.write(payload)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp_path = Path(tmp.name)

    new_sha = sha256_file(tmp_path)
    if path.exists():
        old_sha = sha256_file(path)
        if old_sha == new_sha:
            tmp_path.unlink()
            return "REUSED_IDENTICAL", old_sha
        tmp_path.unlink()
        raise RuntimeError(
            f"Refusing to overwrite incompatible frozen artifact: {path}\n"
            f"existing_sha256={old_sha}\nnew_sha256={new_sha}"
        )
    os.replace(tmp_path, path)
    return "WRITTEN", new_sha

def write_json_frozen(path, obj):
    data = (
        json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)
        + "\n"
    ).encode("utf-8")
    return atomic_write_bytes(path, data)

def write_text_frozen(path, text):
    return atomic_write_bytes(path, text.encode("utf-8"))

def write_csv_frozen(path, df):
    return atomic_write_bytes(path, df.to_csv(index=False).encode("utf-8"))

def write_parquet_frozen(path, df, compression="snappy"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.parent / (path.name + ".tmp.parquet")
    if tmp.exists():
        tmp.unlink()
    df.to_parquet(tmp, index=False, engine="pyarrow", compression=compression)
    new_sha = sha256_file(tmp)
    if path.exists():
        old_sha = sha256_file(path)
        if old_sha == new_sha:
            tmp.unlink()
            return "REUSED_IDENTICAL", old_sha
        tmp.unlink()
        raise RuntimeError(f"Refusing to overwrite incompatible frozen artifact: {path}")
    os.replace(tmp, path)
    return "WRITTEN", new_sha

def register_artifact(path, purpose, section):
    path = Path(path)
    if not path.exists():
        return
    ARTIFACT_RECORDS.append({
        "artifact_name": path.name,
        "artifact_path": str(path),
        "artifact_type": path.suffix.lower().lstrip(".") or "file",
        "file_size": int(path.stat().st_size),
        "sha256": sha256_file(path),
        "purpose": purpose,
        "section": str(section),
    })

def detect_master_format(path):
    suffix = Path(path).suffix.lower()
    if suffix in {".csv", ".txt"}:
        return "csv"
    if suffix in {".parquet", ".pq"}:
        return "parquet"
    raise RuntimeError(f"Unsupported frozen-master format: {suffix!r}")

def resolve_master_path(explicit_path, master_dir):
    if explicit_path is not None:
        p = Path(explicit_path)
        if not p.exists():
            raise FileNotFoundError(f"Configured MASTER_DATASET_PATH does not exist: {p}")
        return p
    master_dir = Path(master_dir)
    if not master_dir.exists():
        raise FileNotFoundError(f"Configured MERGED directory does not exist: {master_dir}")
    candidates = sorted(
        [p for p in master_dir.iterdir()
         if p.is_file() and p.suffix.lower() in {".csv", ".parquet", ".pq"}]
    )
    if len(candidates) != 1:
        raise RuntimeError(
            "MASTER_DATASET_PATH is unset and the MERGED directory does not contain "
            f"exactly one supported master file. Candidates: {[p.name for p in candidates]}. "
            "Set MASTER_DATASET_PATH explicitly."
        )
    return candidates[0]

def extract_manifest_value(manifest, key_paths):
    for path in key_paths:
        cur = manifest
        ok = True
        for key in path:
            if not isinstance(cur, dict) or key not in cur:
                ok = False
                break
            cur = cur[key]
        if ok:
            return cur
    return None

def resolve_merge_manifest(explicit_path, audit_dir, master_sha):
    if explicit_path is not None:
        p = Path(explicit_path)
        if not p.exists():
            return None, None, "UNAVAILABLE_CONFIGURED_PATH"
        with p.open("r", encoding="utf-8") as f:
            return p, json.load(f), "AVAILABLE"

    audit_dir = Path(audit_dir)
    if not audit_dir.exists():
        return None, None, "UNAVAILABLE"

    matching = []
    for p in sorted(audit_dir.glob("*.json")):
        try:
            with p.open("r", encoding="utf-8") as f:
                obj = json.load(f)
        except Exception:
            continue
        candidate_sha = extract_manifest_value(
            obj,
            [
                ("merged", "sha256"),
                ("final_master", "sha256"),
            ],
        )
        if candidate_sha is not None and str(candidate_sha).lower() == master_sha.lower():
            matching.append((p, obj))

    if len(matching) == 1:
        return matching[0][0], matching[0][1], "AVAILABLE"
    if len(matching) == 0:
        return None, None, "UNVERIFIED_NO_HASH_MATCHING_MANIFEST"
    return None, None, "UNVERIFIED_MULTIPLE_HASH_MATCHING_MANIFESTS"

def resolve_source_hash_manifest(explicit_path, audit_dir):
    if explicit_path is not None:
        p = Path(explicit_path)
        return p if p.exists() else None
    audit_dir = Path(audit_dir)
    if not audit_dir.exists():
        return None
    candidates = sorted(
        [p for p in audit_dir.iterdir()
         if p.is_file() and "sha256" in p.name.casefold()]
    )
    return candidates[0] if len(candidates) == 1 else None

def raw_header_and_parser_columns(path):
    fmt = detect_master_format(path)
    if fmt == "csv":
        with Path(path).open("r", encoding="utf-8-sig", errors="strict", newline="") as f:
            raw_header = next(csv.reader(f))
        parsed = list(pd.read_csv(path, nrows=0).columns)
        return raw_header, parsed
    pf = pq.ParquetFile(path)
    cols = list(pf.schema_arrow.names)
    return cols, cols

def iter_master(path, columns=None, chunksize=50000, dtype=None,
                keep_default_na=True, na_filter=True):
    fmt = detect_master_format(path)
    if fmt == "csv":
        for chunk in pd.read_csv(
            path,
            usecols=columns,
            chunksize=chunksize,
            low_memory=False,
            dtype=dtype,
            keep_default_na=keep_default_na,
            na_filter=na_filter,
        ):
            yield chunk
    else:
        pf = pq.ParquetFile(path)
        for batch in pf.iter_batches(batch_size=chunksize, columns=columns):
            yield batch.to_pandas()

def count_master_rows(path, first_column, chunksize):
    if detect_master_format(path) == "parquet":
        return int(pq.ParquetFile(path).metadata.num_rows)
    total = 0
    for chunk in iter_master(path, columns=[first_column], chunksize=chunksize):
        total += len(chunk)
    return total

def exact_cardinality_batches(path, columns, batch_size=4):
    result = {}
    for start in range(0, len(columns), batch_size):
        cols = list(columns[start:start + batch_size])
        if detect_master_format(path) == "csv":
            frame = pd.read_csv(path, usecols=cols, low_memory=False)
        else:
            frame = pd.read_parquet(path, columns=cols, engine="pyarrow")
        for c in cols:
            result[c] = int(frame[c].nunique(dropna=False))
        del frame
        gc.collect()
    return result

def canonical_scalar_bytes(value, numeric):
    if numeric:
        try:
            x = float(value)
        except Exception:
            b = str(value).encode("utf-8", errors="surrogatepass")
            return b"X" + len(b).to_bytes(8, "big") + b
        if math.isnan(x):
            return b"N"
        if math.isinf(x):
            return b"P" if x > 0 else b"M"
        return b"F" + struct.pack(">d", x)

    if pd.isna(value):
        return b"N"
    b = str(value).encode("utf-8", errors="surrogatepass")
    return b"S" + len(b).to_bytes(8, "big") + b

def canonical_row_bytes(row, ordered_columns, numeric_columns):
    numeric_set = set(numeric_columns)
    parts = []
    for c in ordered_columns:
        b = canonical_scalar_bytes(row[c], c in numeric_set)
        parts.append(len(b).to_bytes(8, "big"))
        parts.append(b)
    return b"".join(parts)

def normalize_hash_frame(df, ordered_columns, numeric_columns):
    numeric_set = set(numeric_columns)
    out = pd.DataFrame(index=df.index)
    for c in ordered_columns:
        if c in numeric_set:
            out[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
        else:
            out[c] = df[c].astype("string").fillna("<__MISSING__>")
    return out

def hash128_frame(df, ordered_columns, numeric_columns):
    norm = normalize_hash_frame(df, ordered_columns, numeric_columns)
    h1 = pd.util.hash_pandas_object(
        norm, index=False, hash_key="0123456789abcdef"
    ).astype("uint64").to_numpy()
    h2 = pd.util.hash_pandas_object(
        norm, index=False, hash_key="fedcba9876543210"
    ).astype("uint64").to_numpy()
    del norm
    return h1, h2

def format_h128(h1, h2):
    return f"h128:{int(h1):016x}{int(h2):016x}"

def source_ranges_from_verified_merge_manifest(manifest, master_sha, master_rows):
    manifest_sha = extract_manifest_value(
        manifest,
        [("merged", "sha256"), ("final_master", "sha256")],
    )
    if manifest_sha is None or str(manifest_sha).lower() != master_sha.lower():
        return None, "merge manifest does not cryptographically identify this master"

    no_cleaning = manifest.get("no_cleaning_or_label_modification_during_merge")
    if no_cleaning is not True:
        return None, (
            "merge manifest does not explicitly verify "
            "no_cleaning_or_label_modification_during_merge=True"
        )

    entries = manifest.get("csv_files")
    if entries is None:
        entries = manifest.get("raw_csv_files")
    if not isinstance(entries, list) or not entries:
        return None, "ordered source-file entries are unavailable"

    if any(not isinstance(e, dict) or "filename" not in e or "row_count" not in e
           for e in entries):
        return None, "source-file entries lack filename and row_count"

    if all("file_order" in e for e in entries):
        entries = sorted(entries, key=lambda e: int(e["file_order"]))
    elif isinstance(manifest.get("merge"), dict) and manifest["merge"].get("merge_order"):
        order = list(manifest["merge"]["merge_order"])
        by_name = {e["filename"]: e for e in entries}
        if set(order) != set(by_name):
            return None, "merge_order does not match source-file entries"
        entries = [by_name[n] for n in order]
    else:
        return None, "source-file order is not verifiable"

    ranges = []
    start = 0
    for i, e in enumerate(entries, start=1):
        n = int(e["row_count"])
        end = start + n
        ranges.append({
            "file_order": int(e.get("file_order", i)),
            "source_file": str(e["filename"]),
            "merged_start_0based": int(start),
            "merged_end_exclusive_0based": int(end),
            "row_count": n,
        })
        start = end

    if start != int(master_rows):
        return None, (
            f"sum of verified source rows ({start}) != master rows ({master_rows})"
        )
    return ranges, None

def provenance_for_ordinals(ordinals, ranges, dataset_prefix):
    ordinals = np.asarray(ordinals, dtype=np.int64)
    ends = np.array([r["merged_end_exclusive_0based"] for r in ranges], dtype=np.int64)
    starts = np.array([r["merged_start_0based"] for r in ranges], dtype=np.int64)
    files = np.array([r["source_file"] for r in ranges], dtype=object)

    idx = np.searchsorted(ends, ordinals, side="right")
    if np.any(idx >= len(ranges)):
        raise RuntimeError("Row ordinal exceeded verified source provenance ranges.")

    src_files = files[idx]
    src_rows = ordinals - starts[idx] + 1
    row_ids = np.array([
        f"{dataset_prefix}::{f}::{int(r):010d}"
        for f, r in zip(src_files, src_rows)
    ], dtype=object)
    return src_files, src_rows.astype(np.int64), row_ids

def write_chunked_parquet(path, dataframe_iterator, compression="snappy"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.parent / (path.name + ".tmp.parquet")
    if tmp.exists():
        tmp.unlink()

    writer = None
    wrote = False
    try:
        for df in dataframe_iterator:
            table = pa.Table.from_pandas(df, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(tmp, table.schema, compression=compression)
            writer.write_table(table)
            wrote = True
        if writer is not None:
            writer.close()
            writer = None
        if not wrote:
            raise RuntimeError(f"No rows were written to {path}")

        new_sha = sha256_file(tmp)
        if path.exists():
            old_sha = sha256_file(path)
            if old_sha == new_sha:
                tmp.unlink()
                return "REUSED_IDENTICAL", old_sha
            tmp.unlink()
            raise RuntimeError(f"Refusing to overwrite incompatible frozen artifact: {path}")
        os.replace(tmp, path)
        return "WRITTEN", new_sha
    finally:
        if writer is not None:
            writer.close()
        if tmp.exists():
            tmp.unlink()

def run_duplicate_audit(path, columns, numeric_columns, scratch_dir, chunksize):
    """
    Candidate groups use two deterministic 64-bit pandas row hashes.
    Every candidate duplicate group is then re-read and verified using canonical
    raw-value bytes. SHA-256 resolves any effective-128-bit candidate collision.
    """
    scratch_dir = Path(scratch_dir)
    scratch_dir.mkdir(parents=True, exist_ok=True)
    base_path = scratch_dir / "candidate_hashes.parquet"

    offset = 0
    def chunks():
        nonlocal offset
        for chunk in iter_master(path, columns=columns, chunksize=chunksize):
            h1, h2 = hash128_frame(chunk, columns, numeric_columns)
            n = len(chunk)
            out = pd.DataFrame({
                "row_ordinal_0based": np.arange(offset, offset+n, dtype=np.int64),
                "h1": h1,
                "h2": h2,
            })
            offset += n
            yield out
            del chunk, out, h1, h2
            gc.collect()

    write_chunked_parquet(base_path, chunks())
    base = pd.read_parquet(base_path)

    candidate_sizes = (
        base.groupby(["h1", "h2"], sort=False)
        .size()
        .rename("candidate_size")
        .reset_index()
    )
    dup_keys_df = candidate_sizes[candidate_sizes["candidate_size"] > 1][["h1", "h2"]]
    if dup_keys_df.empty:
        base_path.unlink(missing_ok=True)
        return {
            "duplicate_rows_beyond_first": 0,
            "duplicate_rows_all_occurrences": 0,
            "candidate_duplicate_hash_groups": 0,
            "resolved_effective_128bit_collisions": 0,
            "direct_raw_value_verification": "PASS",
        }

    dup_index = pd.MultiIndex.from_frame(dup_keys_df)
    pair_exemplar = {}
    strong_exemplar = {}
    strong_counts = Counter()
    collision_pairs = set()

    for chunk in iter_master(path, columns=columns, chunksize=chunksize):
        h1, h2 = hash128_frame(chunk, columns, numeric_columns)
        idx = pd.MultiIndex.from_arrays([h1, h2])
        mask = idx.isin(dup_index)

        for pos in np.flatnonzero(mask):
            pair = (int(h1[pos]), int(h2[pos]))
            raw = canonical_row_bytes(
                chunk.iloc[int(pos)], columns, set(numeric_columns)
            )
            if pair not in pair_exemplar:
                pair_exemplar[pair] = raw
            elif pair_exemplar[pair] != raw:
                collision_pairs.add(pair)

            digest = hashlib.sha256(raw).hexdigest()
            skey = (pair, digest)
            if skey not in strong_exemplar:
                strong_exemplar[skey] = raw
            elif strong_exemplar[skey] != raw:
                raise RuntimeError(
                    "Unresolved SHA-256 collision detected during duplicate verification."
                )
            strong_counts[skey] += 1

        del chunk, h1, h2, idx
        gc.collect()

    base_path.unlink(missing_ok=True)
    return {
        "duplicate_rows_beyond_first": int(
            sum(max(0, n-1) for n in strong_counts.values())
        ),
        "duplicate_rows_all_occurrences": int(
            sum(n for n in strong_counts.values() if n > 1)
        ),
        "candidate_duplicate_hash_groups": int(len(dup_keys_df)),
        "resolved_effective_128bit_collisions": int(len(collision_pairs)),
        "direct_raw_value_verification": "PASS",
    }


# Section 1 — Shared Experiment 1 v2 global protocol


In [5]:
# ============================================================
# 7. SECTION 1 — GLOBAL PROTOCOL VERIFICATION / FREEZE
# ============================================================
for d in [
    GLOBAL_DIR, OUTPUT_DIR, MANIFESTS_DIR, AUDITS_DIR, TAXONOMY_DIR,
    IDENTITY_DIR, FEATURE_POLICY_DIR, FINGERPRINTS_DIR, READINESS_DIR, SCRATCH_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

GLOBAL_PROTOCOL_PATH = GLOBAL_DIR / "experiment1_v2_global_protocol.json"

GLOBAL_PROTOCOL = {
    "protocol_name": "Experiment 1 v2",
    "protocol_version": PROTOCOL_VERSION,
    "primary_datasets": ["CICIDS2017", "UNSW-NB15"],
    "primary_task": (
        "Known/documented network-attack detection using reconstruction-based "
        "Autoencoder (AE) and Denoising Autoencoder (DAE)."
    ),
    "benign_only_weight_training": True,
    "stage_independence": [
        "training",
        "early_stopping",
        "threshold_calibration",
        "development_model_selection",
        "final_testing",
    ],
    "attack_family_allowed_uses": [
        "attack_family_aware_partitioning",
        "diagnostics",
        "family_level_reporting",
    ],
    "attack_family_prohibited_uses": [
        "AE_DAE_model_input",
        "imputation_fitting",
        "scaler_fitting",
        "categorical_encoder_fitting",
        "reconstruction_training",
        "threshold_fitting",
    ],
    "AE_DAE_comparison_rules": {
        "common_seeds": True,
        "seed_values_status": "pending later pre-modeling contract freeze",
        "equal_nominal_AE_DAE_search_budget": True,
        "equal_unique_completed_configuration_budget": True,
        "matched_core_architecture_optimization_spaces": True,
        "matched_denoising_ablation": True,
        "same_development_partition_for_AE_DAE": True,
        "same_final_partition_for_AE_DAE": True,
    },
    "post_hoc_change_prohibitions": [
        "partition_membership",
        "feature_policy",
        "taxonomy",
        "primary_threshold",
        "selection_metric",
        "seed_schedule_once_frozen",
        "search_budget",
    ],
    "current_protocol_stage": "Sections 1-7 pre-partition governance/audit",
    "created_by_notebook": "PRE_00_*_Governance_Audit_ModelView.ipynb shared specification",
    "software_environment": {
        "runtime_specific_environment_captured_separately": True
    },
}

substantive = dict(GLOBAL_PROTOCOL)
GLOBAL_PROTOCOL_HASH = sha256_bytes(canonical_json_bytes(substantive))
GLOBAL_PROTOCOL["canonical_protocol_hash"] = GLOBAL_PROTOCOL_HASH

if GLOBAL_PROTOCOL_PATH.exists():
    with GLOBAL_PROTOCOL_PATH.open("r", encoding="utf-8") as f:
        existing = json.load(f)
    existing_substantive = dict(existing)
    existing_substantive.pop("canonical_protocol_hash", None)
    existing_hash = sha256_bytes(canonical_json_bytes(existing_substantive))
    if existing_hash != GLOBAL_PROTOCOL_HASH:
        record_failure(
            1,
            "Existing global protocol conflicts with the required substantive "
            f"Experiment 1 v2 protocol. existing={existing_hash}, "
            f"required={GLOBAL_PROTOCOL_HASH}. The existing file was not overwritten."
        )
    else:
        print("PASS: existing global protocol is substantively compatible.")
else:
    write_json_frozen(GLOBAL_PROTOCOL_PATH, GLOBAL_PROTOCOL)
    print("Created:", GLOBAL_PROTOCOL_PATH)

ENVIRONMENT_MANIFEST_PATH = MANIFESTS_DIR / "software_environment.json"
write_json_frozen(
    ENVIRONMENT_MANIFEST_PATH,
    {
        "dataset": DATASET_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "notebook_stage": NOTEBOOK_STAGE,
        **SOFTWARE_ENVIRONMENT,
    },
)

if SECTION_STATUS["1"] != "FAIL":
    pass_section(1)

print("Global protocol hash:", GLOBAL_PROTOCOL_HASH)


PASS: existing global protocol is substantively compatible.
Global protocol hash: 55713fd82c01168e0a906d282644b1c8dff88632dcb82512c5f62c637e35ff1f


# Section 2 — Dataset provenance and merged-master freeze


In [6]:
# ============================================================
# 8. SECTION 2 — MERGED-MASTER PROVENANCE FREEZE
# ============================================================
try:
    MASTER_DATASET_PATH = resolve_master_path(
        MASTER_DATASET_PATH, MASTER_DATASET_DIR
    )
except Exception as exc:
    record_failure(2, str(exc))
    raise

MASTER_FORMAT = detect_master_format(MASTER_DATASET_PATH)
MASTER_SHA256 = sha256_file(MASTER_DATASET_PATH)
MASTER_FILE_SIZE = int(MASTER_DATASET_PATH.stat().st_size)

RAW_HEADER, PARSER_COLUMNS = raw_header_and_parser_columns(MASTER_DATASET_PATH)
if len(set(PARSER_COLUMNS)) != len(PARSER_COLUMNS):
    record_failure(2, "Parser-visible master column names are not unique.")

MASTER_ROW_COUNT = count_master_rows(
    MASTER_DATASET_PATH, PARSER_COLUMNS[0], CHUNK_SIZE
)
MASTER_COLUMN_COUNT = len(PARSER_COLUMNS)

MERGE_MANIFEST_PATH, MERGE_MANIFEST, MERGE_MANIFEST_STATUS = (
    resolve_merge_manifest(MERGE_MANIFEST_PATH, AUDIT_SEARCH_DIR, MASTER_SHA256)
)
SOURCE_HASH_MANIFEST_PATH = resolve_source_hash_manifest(
    SOURCE_HASH_MANIFEST_PATH, AUDIT_SEARCH_DIR
)

MERGE_MANIFEST_SHA256 = (
    sha256_file(MERGE_MANIFEST_PATH) if MERGE_MANIFEST_PATH is not None else None
)
SOURCE_HASH_MANIFEST_SHA256 = (
    sha256_file(SOURCE_HASH_MANIFEST_PATH)
    if SOURCE_HASH_MANIFEST_PATH is not None else None
)

print("Frozen master:", MASTER_DATASET_PATH)
print("SHA-256      :", MASTER_SHA256)
print("Rows         :", f"{MASTER_ROW_COUNT:,}")
print("Columns      :", MASTER_COLUMN_COUNT)
print("Merge manifest status:", MERGE_MANIFEST_STATUS)

LABEL_COLUMN, label_matches = resolve_unique_column(PARSER_COLUMNS, "Label")
if LABEL_COLUMN is None:
    record_failure(
        2, f"CICIDS2017 Label field is unresolved/ambiguous. Matches={label_matches}"
    )

provenance_policy_status = "UNVERIFIED"
archive_evidence = "UNAVAILABLE"
csv_evidence = "UNAVAILABLE"
source_order = "UNAVAILABLE"

if MERGE_MANIFEST is None:
    record_failure(
        2,
        "No unique merge/provenance manifest cryptographically matches the master; "
        "official MachineLearningCSV provenance cannot be claimed."
    )
else:
    policy = str(MERGE_MANIFEST.get("source_policy", ""))
    if "machinelearningcsv" not in policy.casefold():
        record_failure(
            2,
            "Supplied provenance does not establish the required CICIDS2017 "
            "MachineLearningCSV source policy."
        )
    else:
        provenance_policy_status = "VERIFIED_FROM_SUPPLIED_MERGE_MANIFEST"

    if isinstance(MERGE_MANIFEST.get("zip"), dict):
        archive_evidence = MERGE_MANIFEST["zip"]
    if isinstance(MERGE_MANIFEST.get("csv_files"), list):
        csv_evidence = MERGE_MANIFEST["csv_files"]
        entries = MERGE_MANIFEST["csv_files"]
        if all("file_order" in x for x in entries):
            source_order = [
                x["filename"] for x in sorted(
                    entries, key=lambda x: int(x["file_order"])
                )
            ]

MASTER_MANIFEST = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "notebook_stage": NOTEBOOK_STAGE,
    "created_at_utc": EXECUTION_UTC,
    "required_source_policy": "official CICIDS2017 MachineLearningCSV representation",
    "provenance_policy_status": provenance_policy_status,
    "master_filename": MASTER_DATASET_PATH.name,
    "master_resolved_path": str(MASTER_DATASET_PATH.resolve()),
    "master_sha256": MASTER_SHA256,
    "file_size_bytes": MASTER_FILE_SIZE,
    "row_count": int(MASTER_ROW_COUNT),
    "column_count": int(MASTER_COLUMN_COUNT),
    "column_names_in_order": PARSER_COLUMNS,
    "raw_serialized_header_in_order": RAW_HEADER,
    "label_fields": [LABEL_COLUMN] if LABEL_COLUMN else [],
    "taxonomy_fields": [LABEL_COLUMN] if LABEL_COLUMN else [],
    "merge_manifest_status": MERGE_MANIFEST_STATUS,
    "merge_manifest_path": str(MERGE_MANIFEST_PATH) if MERGE_MANIFEST_PATH else None,
    "merge_manifest_sha256": (
        sha256_file(MERGE_MANIFEST_PATH) if MERGE_MANIFEST_PATH else None
    ),
    "source_hash_manifest_path": (
        str(SOURCE_HASH_MANIFEST_PATH) if SOURCE_HASH_MANIFEST_PATH else None
    ),
    "source_hash_manifest_sha256": (
        sha256_file(SOURCE_HASH_MANIFEST_PATH)
        if SOURCE_HASH_MANIFEST_PATH else None
    ),
    "source_archive_identity_hash_evidence": archive_evidence,
    "source_csv_hash_evidence": csv_evidence,
    "source_file_order": source_order,
    "audit_artifact_locations": {
        "prior_merge_audit_dir": str(AUDIT_SEARCH_DIR),
        "current_preflight_output_dir": str(OUTPUT_DIR),
    },
}
MASTER_MANIFEST_PATH = MANIFESTS_DIR / "CICIDS2017_master_manifest.json"
write_json_frozen(MASTER_MANIFEST_PATH, MASTER_MANIFEST)

if SECTION_STATUS["2"] != "FAIL":
    pass_section(2)

display(pd.DataFrame([{
    "master": MASTER_DATASET_PATH.name,
    "rows": MASTER_ROW_COUNT,
    "columns": MASTER_COLUMN_COUNT,
    "sha256": MASTER_SHA256,
    "provenance_policy_status": provenance_policy_status,
    "section_2": SECTION_STATUS["2"],
}]))


Frozen master: /content/drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/MERGED/CICIDS2017_MachineLearningCSV_merged_raw.csv
SHA-256      : b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff60623871155d51cabd3
Rows         : 2,830,743
Columns      : 79
Merge manifest status: AVAILABLE


,master,rows,columns,sha256,provenance_policy_status,section_2
0,CICIDS2017_MachineLearningCSV_merged_raw.csv,2830743,79,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,VERIFIED_FROM_SUPPLIED_MERGE_MANIFEST,PASS


# Section 3 — Post-merge structural and data-quality audit

> **Implementation correction in this version:** the full-record duplicate-audit
> chunk generator is invoked as `chunks()` rather than passing the generator
> function object. The scientific duplicate definition and source data are unchanged.


In [7]:
COLUMN_ORIGIN_CLASS = {c: "raw_source" for c in PARSER_COLUMNS}
FULL_RECORD_GOVERNANCE_EXCLUSIONS = set()


# ============================================================
# 9. SECTION 3 — STRUCTURAL / DATA-QUALITY AUDIT
# ============================================================
raw_header_counts = Counter(RAW_HEADER)
duplicate_raw_headers = sorted(
    [name for name, n in raw_header_counts.items() if n > 1]
)
if duplicate_raw_headers:
    record_warning(
        3,
        "Duplicate names exist in the serialized raw header. They remain distinct "
        "only through parser-visible positional disambiguation and are explicitly "
        f"recorded: {duplicate_raw_headers}"
    )

mangled_candidates = []
for c in PARSER_COLUMNS:
    m = re.match(r"^(.*)\.(\d+)$", str(c))
    if m and m.group(1) in PARSER_COLUMNS:
        mangled_candidates.append(c)
if mangled_candidates:
    record_warning(
        3,
        f"Parser-visible names consistent with duplicate-header mangling: {mangled_candidates}"
    )

stats = {
    c: {
        "missing": 0,
        "nonmissing": 0,
        "numeric_fail": 0,
        "posinf": 0,
        "neginf": 0,
        "dtypes": set(),
    }
    for c in PARSER_COLUMNS
}

scan_rows = 0
for chunk in iter_master(
    MASTER_DATASET_PATH, columns=PARSER_COLUMNS, chunksize=CHUNK_SIZE
):
    scan_rows += len(chunk)
    for c in PARSER_COLUMNS:
        s = chunk[c]
        st = stats[c]
        st["dtypes"].add(str(s.dtype))
        miss = s.isna()
        st["missing"] += int(miss.sum())
        st["nonmissing"] += int((~miss).sum())

        num = pd.to_numeric(s, errors="coerce")
        st["numeric_fail"] += int((num.isna() & ~miss).sum())
        arr = num.to_numpy(dtype=np.float64, na_value=np.nan)
        st["posinf"] += int(np.isposinf(arr).sum())
        st["neginf"] += int(np.isneginf(arr).sum())

    del chunk
    gc.collect()

if scan_rows != MASTER_ROW_COUNT:
    record_failure(
        3,
        f"Structural audit row count {scan_rows} != provenance row count {MASTER_ROW_COUNT}."
    )

CARDINALITY = exact_cardinality_batches(
    MASTER_DATASET_PATH,
    PARSER_COLUMNS,
    batch_size=CARDINALITY_COLUMN_BATCH_SIZE,
)

# Exact dominant frequency is only needed for columns with <= the descriptive
# near-constant unique-value cap.
low_card_columns = [
    c for c in PARSER_COLUMNS if CARDINALITY[c] <= NEAR_CONSTANT_MAX_UNIQUE
]
dominant = {}
if low_card_columns:
    counters = {c: Counter() for c in low_card_columns}
    for chunk in iter_master(
        MASTER_DATASET_PATH, columns=low_card_columns, chunksize=CHUNK_SIZE
    ):
        for c in low_card_columns:
            counters[c].update(chunk[c].value_counts(dropna=False).to_dict())
        del chunk
        gc.collect()
    for c in low_card_columns:
        if counters[c]:
            v, n = counters[c].most_common(1)[0]
            dominant[c] = (v, int(n))

column_rows = []
near_rows = []

for i, c in enumerate(PARSER_COLUMNS, start=1):
    st = stats[c]
    numeric_parseable = (
        st["nonmissing"] > 0 and st["numeric_fail"] == 0
    )
    dom_v, dom_n = dominant.get(c, (None, None))
    dom_frac = (
        float(dom_n) / float(MASTER_ROW_COUNT)
        if dom_n is not None and MASTER_ROW_COUNT else None
    )
    is_constant = CARDINALITY[c] == 1
    is_near = (
        CARDINALITY[c] > 1
        and CARDINALITY[c] <= NEAR_CONSTANT_MAX_UNIQUE
        and dom_frac is not None
        and dom_frac >= NEAR_CONSTANT_DOMINANT_FRACTION
    )

    column_rows.append({
        "dataset": DATASET_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "master_dataset_sha256": MASTER_SHA256,
        "created_at_utc": EXECUTION_UTC,
        "notebook_stage": NOTEBOOK_STAGE,
        "column_position_1based": i,
        "column_name": c,
        "raw_serialized_header_name": RAW_HEADER[i-1] if i-1 < len(RAW_HEADER) else None,
        "column_origin_class": COLUMN_ORIGIN_CLASS.get(c, "raw_source"),
        "observed_dtypes": json.dumps(sorted(st["dtypes"])),
        "numeric_parseable_full_corpus": bool(numeric_parseable),
        "numeric_conversion_failure_count": int(st["numeric_fail"]),
        "pandas_missing_or_nan_count": int(st["missing"]),
        "numeric_nan_count": int(st["missing"]) if numeric_parseable else None,
        "positive_inf_count": int(st["posinf"]),
        "negative_inf_count": int(st["neginf"]),
        "cardinality_including_missing": int(CARDINALITY[c]),
    })

    near_rows.append({
        "dataset": DATASET_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "master_dataset_sha256": MASTER_SHA256,
        "created_at_utc": EXECUTION_UTC,
        "notebook_stage": NOTEBOOK_STAGE,
        "column_name": c,
        "cardinality_including_missing": int(CARDINALITY[c]),
        "descriptive_max_unique_criterion": int(NEAR_CONSTANT_MAX_UNIQUE),
        "descriptive_dominant_fraction_criterion": float(
            NEAR_CONSTANT_DOMINANT_FRACTION
        ),
        "dominant_value_repr_if_low_cardinality": (
            repr(dom_v) if dom_n is not None else None
        ),
        "dominant_count_if_low_cardinality": dom_n,
        "dominant_fraction_if_low_cardinality": dom_frac,
        "constant": bool(is_constant),
        "near_constant_descriptive_only": bool(is_near),
    })

COLUMN_AUDIT_DF = pd.DataFrame(column_rows)
NEAR_CONSTANT_DF = pd.DataFrame(near_rows)

NUMERIC_COLUMNS_OBSERVED = COLUMN_AUDIT_DF.loc[
    COLUMN_AUDIT_DF["numeric_parseable_full_corpus"], "column_name"
].tolist()
NONNUMERIC_COLUMNS_OBSERVED = COLUMN_AUDIT_DF.loc[
    ~COLUMN_AUDIT_DF["numeric_parseable_full_corpus"], "column_name"
].tolist()

# Identifier review: a name signal alone is insufficient.
# The notebook flags a field only when a name signal is accompanied by very high cardinality.
identifier_rows = []
for row in COLUMN_AUDIT_DF.itertuples(index=False):
    name_key = norm_name(row.column_name)
    name_signal = any(
        token in name_key
        for token in ["srcip", "dstip", "sourceip", "destinationip", "flowid",
                      "recordid", "rowid", "timestamp"]
    )
    high_card = (
        int(row.cardinality_including_missing)
        >= max(1000, int(0.95 * MASTER_ROW_COUNT))
    )
    if name_signal and high_card:
        identifier_rows.append({
            "dataset": DATASET_NAME,
            "column_name": row.column_name,
            "review_status": "SUSPECTED_IDENTIFIER_REVIEW_ONLY",
            "evidence": (
                "identifier-like name signal plus cardinality >=95% of row count; "
                "this audit alone does not exclude the field"
            ),
            "cardinality": int(row.cardinality_including_missing),
        })

IDENTIFIER_REVIEW_DF = pd.DataFrame(
    identifier_rows,
    columns=[
        "dataset", "column_name", "review_status", "evidence", "cardinality"
    ],
)

# Full-record duplicate audit: substantive source fields only.
SUBSTANTIVE_COLUMNS = [
    c for c in PARSER_COLUMNS if c not in FULL_RECORD_GOVERNANCE_EXCLUSIONS
]
SUBSTANTIVE_NUMERIC = [
    c for c in SUBSTANTIVE_COLUMNS if c in NUMERIC_COLUMNS_OBSERVED
]

try:
    FULL_RECORD_DUPLICATE_AUDIT = run_duplicate_audit(
        MASTER_DATASET_PATH,
        SUBSTANTIVE_COLUMNS,
        SUBSTANTIVE_NUMERIC,
        SCRATCH_DIR / "section3_full_record_duplicates",
        CHUNK_SIZE,
    )
except Exception as exc:
    FULL_RECORD_DUPLICATE_AUDIT = {"status": "FAILED", "error": str(exc)}
    record_failure(3, f"Full-record exact-duplicate audit failed: {exc}")

SCHEMA_AUDIT_PATH = AUDITS_DIR / "postmerge_schema_audit.csv"
CARDINALITY_AUDIT_PATH = AUDITS_DIR / "column_cardinality_audit.csv"
NEARCONST_PATH = AUDITS_DIR / "constant_nearconstant_audit.csv"
IDENTIFIER_PATH = AUDITS_DIR / "identifier_review.csv"
QUALITY_PATH = AUDITS_DIR / "postmerge_quality_audit.json"
SUMMARY_MD_PATH = AUDITS_DIR / "postmerge_audit_summary.md"

write_csv_frozen(SCHEMA_AUDIT_PATH, COLUMN_AUDIT_DF)
write_csv_frozen(
    CARDINALITY_AUDIT_PATH,
    COLUMN_AUDIT_DF[
        ["dataset", "column_position_1based", "column_name",
         "cardinality_including_missing"]
    ],
)
write_csv_frozen(NEARCONST_PATH, NEAR_CONSTANT_DF)
write_csv_frozen(IDENTIFIER_PATH, IDENTIFIER_REVIEW_DF)

QUALITY_SUMMARY = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "master_dataset_sha256": MASTER_SHA256,
    "created_at_utc": EXECUTION_UTC,
    "notebook_stage": NOTEBOOK_STAGE,
    "row_count": int(MASTER_ROW_COUNT),
    "column_count": int(MASTER_COLUMN_COUNT),
    "duplicate_raw_headers": duplicate_raw_headers,
    "parser_mangled_header_candidates": mangled_candidates,
    "numeric_columns_observed": NUMERIC_COLUMNS_OBSERVED,
    "nonnumeric_columns_observed": NONNUMERIC_COLUMNS_OBSERVED,
    "total_pandas_missing_or_nan_cells": int(
        COLUMN_AUDIT_DF["pandas_missing_or_nan_count"].sum()
    ),
    "total_positive_inf_cells": int(
        COLUMN_AUDIT_DF["positive_inf_count"].sum()
    ),
    "total_negative_inf_cells": int(
        COLUMN_AUDIT_DF["negative_inf_count"].sum()
    ),
    "full_record_duplicate_audit": FULL_RECORD_DUPLICATE_AUDIT,
    "near_constant_criterion": {
        "max_unique_including_missing": int(NEAR_CONSTANT_MAX_UNIQUE),
        "dominant_fraction_threshold": float(NEAR_CONSTANT_DOMINANT_FRACTION),
        "use": "descriptive only; not a learned feature-selection threshold",
    },
}
write_json_frozen(QUALITY_PATH, QUALITY_SUMMARY)

summary_text = (
    f"# Post-merge audit summary — {DATASET_NAME}\n\n"
    f"- Rows: {MASTER_ROW_COUNT:,}\n"
    f"- Columns: {MASTER_COLUMN_COUNT}\n"
    f"- Numeric-parseable columns: {len(NUMERIC_COLUMNS_OBSERVED)}\n"
    f"- Non-numeric columns: {len(NONNUMERIC_COLUMNS_OBSERVED)}\n"
    f"- Missing/NaN cells: {QUALITY_SUMMARY['total_pandas_missing_or_nan_cells']:,}\n"
    f"- +Inf cells: {QUALITY_SUMMARY['total_positive_inf_cells']:,}\n"
    f"- -Inf cells: {QUALITY_SUMMARY['total_negative_inf_cells']:,}\n"
    f"- Full-record duplicate rows beyond first: "
    f"{FULL_RECORD_DUPLICATE_AUDIT.get('duplicate_rows_beyond_first', 'UNAVAILABLE')}\n"
    f"- Duplicate raw header names: {duplicate_raw_headers}\n"
)
write_text_frozen(SUMMARY_MD_PATH, summary_text)

if SECTION_STATUS["3"] != "FAIL":
    pass_section(3)

print("Dataset shape:", (MASTER_ROW_COUNT, MASTER_COLUMN_COUNT))
display(COLUMN_AUDIT_DF.head(30))
display(
    NEAR_CONSTANT_DF[
        NEAR_CONSTANT_DF["constant"]
        | NEAR_CONSTANT_DF["near_constant_descriptive_only"]
    ].head(30)
)


DEST_PORT_COLUMN, dest_matches = resolve_unique_column(
    PARSER_COLUMNS, "Destination Port"
)
if DEST_PORT_COLUMN is None:
    record_failure(
        3,
        f"Destination Port is unresolved/ambiguous. Matches={dest_matches}"
    )
else:
    row = COLUMN_AUDIT_DF[
        COLUMN_AUDIT_DF["column_name"].eq(DEST_PORT_COLUMN)
    ].iloc[0]
    if not bool(row["numeric_parseable_full_corpus"]):
        record_failure(
            3,
            "Destination Port is not numerically parseable across the full master."
        )

if LABEL_COLUMN is not None:
    candidate_cols = [c for c in PARSER_COLUMNS if c != LABEL_COLUMN]
    unexpected_non_numeric = [
        c for c in candidate_cols if c not in NUMERIC_COLUMNS_OBSERVED
    ]
    if unexpected_non_numeric:
        record_failure(
            3,
            "Unexpected nonnumeric CICIDS2017 candidate feature columns: "
            f"{unexpected_non_numeric}"
        )

label_counter = Counter()
for chunk in iter_master(
    MASTER_DATASET_PATH,
    columns=[LABEL_COLUMN],
    chunksize=CHUNK_SIZE,
    dtype={LABEL_COLUMN: "string"},
    keep_default_na=False,
    na_filter=False,
):
    label_counter.update(chunk[LABEL_COLUMN].astype(str).tolist())

LABEL_AUDIT_DF = pd.DataFrame([
    {
        "dataset": DATASET_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "master_dataset_sha256": MASTER_SHA256,
        "created_at_utc": EXECUTION_UTC,
        "notebook_stage": NOTEBOOK_STAGE,
        "field": LABEL_COLUMN,
        "raw_value": value,
        "display_repr": repr(value),
        "support": int(n),
    }
    for value, n in label_counter.items()
]).sort_values("support", ascending=False)

LABEL_AUDIT_PATH = AUDITS_DIR / "postmerge_label_audit.csv"
write_csv_frozen(LABEL_AUDIT_PATH, LABEL_AUDIT_DF)

print("CICIDS2017 label distribution:")
display(LABEL_AUDIT_DF)


[SECTION 3 WARNING] Duplicate names exist in the serialized raw header. They remain distinct only through parser-visible positional disambiguation and are explicitly recorded: [' Fwd Header Length']
[SECTION 3 WARNING] Parser-visible names consistent with duplicate-header mangling: [' Fwd Header Length.1']
Dataset shape: (2830743, 79)


,dataset,protocol_version,master_dataset_sha256,created_at_utc,notebook_stage,column_position_1based,column_name,raw_serialized_header_name,column_origin_class,observed_dtypes,numeric_parseable_full_corpus,numeric_conversion_failure_count,pandas_missing_or_nan_count,numeric_nan_count,positive_inf_count,negative_inf_count,cardinality_including_missing
0,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,1,Destination Port,Destination Port,raw_source,"[""int64""]",True,0,0,0.0,0,0,53805
1,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,2,Flow Duration,Flow Duration,raw_source,"[""int64""]",True,0,0,0.0,0,0,1050899
2,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,3,Total Fwd Packets,Total Fwd Packets,raw_source,"[""int64""]",True,0,0,0.0,0,0,1432
3,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,4,Total Backward Packets,Total Backward Packets,raw_source,"[""int64""]",True,0,0,0.0,0,0,1747
4,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,5,Total Length of Fwd Packets,Total Length of Fwd Packets,raw_source,"[""int64""]",True,0,0,0.0,0,0,17928
5,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,6,Total Length of Bwd Packets,Total Length of Bwd Packets,raw_source,"[""int64""]",True,0,0,0.0,0,0,64698
6,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,7,Fwd Packet Length Max,Fwd Packet Length Max,raw_source,"[""int64""]",True,0,0,0.0,0,0,5279
7,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,8,Fwd Packet Length Min,Fwd Packet Length Min,raw_source,"[""int64""]",True,0,0,0.0,0,0,384
8,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,9,Fwd Packet Length Mean,Fwd Packet Length Mean,raw_source,"[""float64""]",True,0,0,0.0,0,0,99716
9,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,10,Fwd Packet Length Std,Fwd Packet Length Std,raw_source,"[""float64""]",True,0,0,0.0,0,0,253909


,dataset,protocol_version,master_dataset_sha256,created_at_utc,notebook_stage,column_name,cardinality_including_missing,descriptive_max_unique_criterion,descriptive_dominant_fraction_criterion,dominant_value_repr_if_low_cardinality,dominant_count_if_low_cardinality,dominant_fraction_if_low_cardinality,constant,near_constant_descriptive_only
31,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Bwd PSH Flags,1,20,0.999,0,2830743.0,1.000000,True,False
32,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Fwd URG Flags,2,20,0.999,0,2830428.0,0.999889,False,True
33,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Bwd URG Flags,1,20,0.999,0,2830743.0,1.000000,True,False
45,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,RST Flag Count,2,20,0.999,0,2830057.0,0.999758,False,True
49,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,CWE Flag Count,2,20,0.999,0,2830428.0,0.999889,False,True
50,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,ECE Flag Count,2,20,0.999,0,2830054.0,0.999757,False,True
56,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Fwd Avg Bytes/Bulk,1,20,0.999,0,2830743.0,1.000000,True,False
57,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Fwd Avg Packets/Bulk,1,20,0.999,0,2830743.0,1.000000,True,False
58,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Fwd Avg Bulk Rate,1,20,0.999,0,2830743.0,1.000000,True,False
59,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Bwd Avg Bytes/Bulk,1,20,0.999,0,2830743.0,1.000000,True,False


CICIDS2017 label distribution:


,dataset,protocol_version,master_dataset_sha256,created_at_utc,notebook_stage,field,raw_value,display_repr,support
0,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,BENIGN,'BENIGN',2273097
5,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,DoS Hulk,'DoS Hulk',231073
13,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,PortScan,'PortScan',158930
14,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,DDoS,'DDoS',128027
6,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,DoS GoldenEye,'DoS GoldenEye',10293
1,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,FTP-Patator,'FTP-Patator',7938
2,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,SSH-Patator,'SSH-Patator',5897
3,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,DoS slowloris,'DoS slowloris',5796
4,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,DoS Slowhttptest,'DoS Slowhttptest',5499
12,CICIDS2017,Experiment_1_v2,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...,2026-09-14T08:23:35.731576+00:00,PRE_00_SECTIONS_1_TO_7,Label,Bot,'Bot',1966


# Section 4 — Taxonomy freeze


In [8]:
# ============================================================
# 10. SECTION 4 — CICIDS2017 TAXONOMY FREEZE
# ============================================================
TAXONOMY_AUDIT_CSV_PATH = TAXONOMY_DIR / "taxonomy_audit.csv"
TAXONOMY_AUDIT_MD_PATH = TAXONOMY_DIR / "taxonomy_audit.md"
TAXONOMY_MANIFEST_PATH = TAXONOMY_DIR / "taxonomy_manifest.json"
TAXONOMY_HASH = None

if LABEL_COLUMN is None:
    record_failure(4, "Taxonomy cannot be frozen because Label is unresolved.")
else:
    values = LABEL_AUDIT_DF[
        LABEL_AUDIT_DF["field"].eq(LABEL_COLUMN)
    ].copy()
    values["trimmed_value"] = values["raw_value"].astype(str).str.strip()

    benign_mask = values["trimmed_value"].str.casefold().eq("benign")
    if int(benign_mask.sum()) != 1:
        record_failure(
            4,
            "CICIDS2017 benign semantics are unresolved: exactly one supplied label "
            "must normalize to BENIGN."
        )

    whitespace_anomalies = values[
        values["raw_value"].astype(str) != values["trimmed_value"]
    ]
    if len(whitespace_anomalies):
        record_failure(
            4,
            "Whitespace/formatting anomalies exist in CICIDS source label values. "
            "No family normalization beyond identity is frozen for CICIDS2017."
        )

    if values["raw_value"].astype(str).eq("").any():
        record_failure(4, "An empty CICIDS label value was observed.")

    taxonomy_rows = []
    for r in values.itertuples(index=False):
        raw = str(r.raw_value)
        benign = raw.strip().casefold() == "benign"
        taxonomy_rows.append({
            "original_value": raw,
            "display_safe_representation": repr(raw),
            "canonical_value": None if benign else raw,
            "binary_label": 0 if benign else 1,
            "original_support": int(r.support),
            "canonical_support": None if benign else int(r.support),
            "transformation_rule": "identity",
            "transformation_reason": (
                "verified supplied benign label" if benign
                else "supplied CICIDS attack label; no family collapsing"
            ),
            "validation_status": (
                "PASS" if raw == raw.strip() and raw != "" else "FAIL"
            ),
        })

    TAXONOMY_AUDIT_DF = pd.DataFrame(taxonomy_rows)

    mapping_spec = {
        "dataset": DATASET_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "source_field": LABEL_COLUMN,
        "benign_semantics": (
            "the unique supplied value whose strip+casefold representation equals 'benign'"
        ),
        "attack_family_mapping": "identity on supplied attack labels",
        "rare_family_collapsing": False,
        "lycos_reconstruction": False,
    }
    TAXONOMY_HASH = sha256_bytes(canonical_json_bytes(mapping_spec))
    TAXONOMY_MANIFEST = {
        **mapping_spec,
        "taxonomy_hash": TAXONOMY_HASH,
        "master_dataset_sha256": MASTER_SHA256,
        "created_at_utc": EXECUTION_UTC,
        "TAXONOMY_FREEZE_STATUS": (
            "PASS" if SECTION_STATUS["4"] != "FAIL" else "FAIL"
        ),
    }

    write_csv_frozen(TAXONOMY_AUDIT_CSV_PATH, TAXONOMY_AUDIT_DF)
    write_text_frozen(
        TAXONOMY_AUDIT_MD_PATH,
        "# CICIDS2017 Taxonomy Audit\n\n"
        + markdown_table(TAXONOMY_AUDIT_DF)
        + "\n",
    )
    write_json_frozen(TAXONOMY_MANIFEST_PATH, TAXONOMY_MANIFEST)

if SECTION_STATUS["4"] != "FAIL":
    pass_section(4)

if "TAXONOMY_AUDIT_DF" in globals():
    display(TAXONOMY_AUDIT_DF)


,original_value,display_safe_representation,canonical_value,binary_label,original_support,canonical_support,transformation_rule,transformation_reason,validation_status
0,BENIGN,'BENIGN',None,0,2273097,NaN,identity,verified supplied benign label,PASS
1,DoS Hulk,'DoS Hulk',DoS Hulk,1,231073,231073.0,identity,supplied CICIDS attack label; no family collap...,PASS
2,PortScan,'PortScan',PortScan,1,158930,158930.0,identity,supplied CICIDS attack label; no family collap...,PASS
3,DDoS,'DDoS',DDoS,1,128027,128027.0,identity,supplied CICIDS attack label; no family collap...,PASS
4,DoS GoldenEye,'DoS GoldenEye',DoS GoldenEye,1,10293,10293.0,identity,supplied CICIDS attack label; no family collap...,PASS
5,FTP-Patator,'FTP-Patator',FTP-Patator,1,7938,7938.0,identity,supplied CICIDS attack label; no family collap...,PASS
6,SSH-Patator,'SSH-Patator',SSH-Patator,1,5897,5897.0,identity,supplied CICIDS attack label; no family collap...,PASS
7,DoS slowloris,'DoS slowloris',DoS slowloris,1,5796,5796.0,identity,supplied CICIDS attack label; no family collap...,PASS
8,DoS Slowhttptest,'DoS Slowhttptest',DoS Slowhttptest,1,5499,5499.0,identity,supplied CICIDS attack label; no family collap...,PASS
9,Bot,'Bot',Bot,1,1966,1966.0,identity,supplied CICIDS attack label; no family collap...,PASS


# Section 5 — Immutable observation identity


In [9]:
# ============================================================
# 11. SECTION 5 — IMMUTABLE OBSERVATION IDENTITY
# ============================================================
ROW_IDENTITY_PATH = IDENTITY_DIR / "row_identity.parquet"
ROW_ID_MANIFEST_PATH = IDENTITY_DIR / "row_identity_manifest.json"
ROW_ID_SPEC_HASH = None
SOURCE_RANGES = None

if MERGE_MANIFEST is None:
    record_failure(
        5,
        "Verified merge provenance is unavailable; row_id will NOT be fabricated "
        "from the current merged DataFrame index."
    )
else:
    SOURCE_RANGES, err = source_ranges_from_verified_merge_manifest(
        MERGE_MANIFEST, MASTER_SHA256, MASTER_ROW_COUNT
    )
    if SOURCE_RANGES is None:
        record_failure(
            5,
            "Trustworthy source_file/source_row_number identity cannot be restored "
            f"from the merge evidence: {err}"
        )

if SECTION_STATUS["5"] != "FAIL":
    ROW_ID_SPEC = {
        "dataset": DATASET_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "master_dataset_sha256": MASTER_SHA256,
        "construction_rule": (
            "dataset_prefix::verified_source_filename::1-based_source_data_row_number"
        ),
        "source_provenance_method": (
            "verified merge-manifest source-file order + exact source row counts, "
            "accepted only after master SHA-256 and no-cleaning assertion verification"
        ),
        "uses_feature_values": False,
        "uses_current_dataframe_index_as_identity": False,
        "source_ranges": SOURCE_RANGES,
    }
    ROW_ID_SPEC_HASH = sha256_bytes(canonical_json_bytes(ROW_ID_SPEC))
    ROW_ID_SPEC["row_identity_specification_hash"] = ROW_ID_SPEC_HASH

    def identity_chunks():
        offset = 0
        while offset < MASTER_ROW_COUNT:
            end = min(offset + ROW_ID_CHUNK_SIZE, MASTER_ROW_COUNT)
            ords = np.arange(offset, end, dtype=np.int64)
            src_files, src_rows, row_ids = provenance_for_ordinals(
                ords, SOURCE_RANGES, DATASET_PREFIX
            )
            yield pd.DataFrame({
                "row_ordinal_0based": ords,
                "row_id": row_ids,
                "source_file": src_files,
                "source_row_number_1based": src_rows,
            })
            offset = end

    try:
        write_chunked_parquet(ROW_IDENTITY_PATH, identity_chunks())

        identity = pd.read_parquet(
            ROW_IDENTITY_PATH,
            columns=["row_id", "source_file", "source_row_number_1based"],
        )
        null_count = int(identity["row_id"].isna().sum())
        unique_count = int(identity["row_id"].nunique(dropna=False))

        if null_count:
            record_failure(5, f"row_id contains {null_count} null values.")
        if unique_count != MASTER_ROW_COUNT:
            record_failure(
                5,
                f"row_id uniqueness failure: unique={unique_count:,}, "
                f"rows={MASTER_ROW_COUNT:,}"
            )

        ROW_ID_MANIFEST = {
            **ROW_ID_SPEC,
            "row_count": int(MASTER_ROW_COUNT),
            "row_id_null_count": null_count,
            "row_id_unique_count": unique_count,
            "validation_status": (
                "PASS"
                if null_count == 0 and unique_count == MASTER_ROW_COUNT
                else "FAIL"
            ),
            "sample_ids_for_audit_only": identity["row_id"].head(5).tolist(),
            "row_identity_artifact": str(ROW_IDENTITY_PATH),
            "row_identity_artifact_sha256": sha256_file(ROW_IDENTITY_PATH),
            "created_at_utc": EXECUTION_UTC,
        }
        write_json_frozen(ROW_ID_MANIFEST_PATH, ROW_ID_MANIFEST)

        display(pd.DataFrame([{
            "rows": MASTER_ROW_COUNT,
            "unique_row_ids": unique_count,
            "null_row_ids": null_count,
            "row_identity_specification_hash": ROW_ID_SPEC_HASH,
        }]))

        del identity
        gc.collect()
    except Exception as exc:
        record_failure(5, f"row identity artifact generation/validation failed: {exc}")

if SECTION_STATUS["5"] != "FAIL":
    pass_section(5)


,rows,unique_row_ids,null_row_ids,row_identity_specification_hash
0,2830743,2830743,0,75d07ee359615a65b65107ba4999df581e382e13395916...


# Section 6 — Raw feature-policy freeze


In [10]:
# ============================================================
# 12. SECTION 6 — CICIDS2017 RAW FEATURE-POLICY FREEZE
# ============================================================
FEATURE_POLICY_PATH = FEATURE_POLICY_DIR / "raw_feature_policy.json"
FEATURE_POLICY_HASH = None
MODEL_COLUMNS = []
NUMERICAL_MODEL_COLUMNS = []
CATEGORICAL_MODEL_COLUMNS = []

if not dependencies_pass(2, 3, 4, 5):
    upstream_status = {
        str(s): SECTION_STATUS[str(s)]
        for s in [2, 3, 4, 5]
    }
    upstream_failures = [
        x for x in BLOCKING_FAILURES
        if x.get("section") in {"2", "3", "4", "5"}
    ]

    print("Upstream Section 2–5 status:", upstream_status)
    if upstream_failures:
        print("Upstream blocking failures:")
        for item in upstream_failures:
            print(
                f"  Section {item['section']}: {item['message']}"
            )

    record_failure(
        6,
        "Feature-policy freeze is blocked because Sections 2–5 did not all PASS. "
        "Resolve the upstream blocking failures shown above; the Section 6 gate "
        "will not be bypassed."
    )
else:
    MODEL_COLUMNS = [c for c in PARSER_COLUMNS if c != LABEL_COLUMN]
    NUMERICAL_MODEL_COLUMNS = list(MODEL_COLUMNS)

    nonnumeric = [
        c for c in MODEL_COLUMNS if c not in NUMERIC_COLUMNS_OBSERVED
    ]
    if nonnumeric:
        record_failure(
            6,
            f"Retained CICIDS model-view contains nonnumeric columns: {nonnumeric}"
        )

    DEST_PORT_COLUMN, _ = resolve_unique_column(PARSER_COLUMNS, "Destination Port")
    if DEST_PORT_COLUMN is None or DEST_PORT_COLUMN not in MODEL_COLUMNS:
        record_failure(
            6,
            "Verified numerical Destination Port is not retained in the raw model view."
        )

    prohibited_norms = {
        norm_name(x)
        for x in [
            LABEL_COLUMN, "row_id", "feature_fingerprint",
            "attack_family", "binary_label", "source_file", "source_row_number"
        ]
    }
    accidental = [c for c in MODEL_COLUMNS if norm_name(c) in prohibited_norms]
    if accidental:
        record_failure(
            6,
            f"Prohibited target/family/identity metadata entered model input: {accidental}"
        )

    exclusions = {
        LABEL_COLUMN: "official label and attack-family metadata; prohibited model input"
    }
    inclusions = {
        c: (
            "verified numerically parseable CICFlowMeter raw traffic feature; "
            "retained prospectively without learned preprocessing"
        )
        for c in MODEL_COLUMNS
    }

    spec = {
        "dataset": DATASET_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "master_dataset_sha256": MASTER_SHA256,
        "included_model_columns": MODEL_COLUMNS,
        "excluded_columns": list(exclusions),
        "metadata_columns": [
            LABEL_COLUMN, "row_id(sidecar)", "source_file(sidecar)",
            "source_row_number(sidecar)"
        ],
        "numerical_model_columns": NUMERICAL_MODEL_COLUMNS,
        "categorical_model_columns": [],
        "exclusion_reason_per_column": exclusions,
        "inclusion_reason_per_column": inclusions,
        "column_order": MODEL_COLUMNS,
        "feature_policy_status": (
            "PASS" if SECTION_STATUS["6"] != "FAIL" else "FAIL"
        ),
        "expected_future_corruption_structure": {
            "numerical_masking": "searched later",
            "categorical_masking": "N/A",
        },
        "learned_preprocessing_fitted_here": False,
    }
    FEATURE_POLICY_HASH = sha256_bytes(canonical_json_bytes(spec))
    spec["feature_policy_hash"] = FEATURE_POLICY_HASH
    write_json_frozen(FEATURE_POLICY_PATH, spec)

    if len(MODEL_COLUMNS) != 78:
        record_warning(
            6,
            f"Verified CICIDS raw numerical model-input count is {len(MODEL_COLUMNS)}, "
            "not the commonly documented 78. The protocol does not force the count."
        )

if SECTION_STATUS["6"] != "FAIL":
    pass_section(6)

if SECTION_STATUS["6"] == "PASS":
    display(pd.DataFrame([
        {"class": "numerical_model_columns", "count": len(NUMERICAL_MODEL_COLUMNS)},
        {"class": "categorical_model_columns", "count": 0},
        {"class": "excluded_source_columns", "count": 1},
    ]))

FINGERPRINT_METADATA_READ_COLUMNS = [LABEL_COLUMN]

def derive_binary_and_family_metadata(chunk):
    raw = chunk[LABEL_COLUMN].astype("string")
    trimmed = raw.str.strip()
    benign = trimmed.str.casefold().eq("benign")
    if benign.isna().any():
        raise RuntimeError("CICIDS binary label derivation encountered unresolved missing labels.")
    return (~benign).astype("int8"), raw.where(~benign, pd.NA)


,class,count
0,numerical_model_columns,78
1,categorical_model_columns,0
2,excluded_source_columns,1


# Section 7 — Model-view fingerprint and duplicate-leakage audit


In [11]:
# ============================================================
# 13. SECTION 7 — MODEL-VIEW FINGERPRINT / DUPLICATE-LEAKAGE AUDIT
# ============================================================
FINGERPRINT_AUDIT = None
FINGERPRINT_MANIFEST_HASH = None

ROW_GROUP_MAPPING_PATH = (
    FINGERPRINTS_DIR / "row_modelview_group_mapping.parquet"
)
DUPLICATE_GROUP_MANIFEST_PATH = (
    FINGERPRINTS_DIR / "duplicate_group_manifest.parquet"
)
CONFLICT_GROUP_MANIFEST_PATH = (
    FINGERPRINTS_DIR / "conflicting_fingerprint_manifest.parquet"
)
MIXED_FAMILY_SUMMARY_PATH = (
    FINGERPRINTS_DIR / "mixed_label_family_summary.csv"
)
FINGERPRINT_AUDIT_PATH = FINGERPRINTS_DIR / "fingerprint_audit.json"

if not dependencies_pass(2, 3, 4, 5, 6):
    record_failure(
        7,
        "Fingerprinting is blocked because Sections 2–6 did not all PASS."
    )
else:
    BASE_PATH = SCRATCH_DIR / "modelview_base.parquet"
    if BASE_PATH.exists():
        BASE_PATH.unlink()

    offset_box = [0]

    def base_chunks():
        read_cols = list(MODEL_COLUMNS)
        for c in FINGERPRINT_METADATA_READ_COLUMNS:
            if c not in read_cols:
                read_cols.append(c)

        for chunk in iter_master(
            MASTER_DATASET_PATH,
            columns=read_cols,
            chunksize=CHUNK_SIZE,
        ):
            h1, h2 = hash128_frame(
                chunk[MODEL_COLUMNS],
                MODEL_COLUMNS,
                NUMERICAL_MODEL_COLUMNS,
            )
            binary, family = derive_binary_and_family_metadata(chunk)
            n = len(chunk)
            start = offset_box[0]

            out = pd.DataFrame({
                "row_ordinal_0based": np.arange(start, start+n, dtype=np.int64),
                "h1": h1,
                "h2": h2,
                "binary_label": binary.astype("int8"),
                "attack_family_metadata": family.astype("string"),
            })
            offset_box[0] = start + n
            yield out

            del chunk, out, h1, h2, binary, family
            gc.collect()

    try:
        write_chunked_parquet(BASE_PATH, base_chunks())
        base = pd.read_parquet(BASE_PATH)
    except Exception as exc:
        record_failure(7, f"Base fingerprint generation failed: {exc}")
        base = None

if SECTION_STATUS["7"] != "FAIL":
    pair_sizes = (
        base.groupby(["h1", "h2"], sort=False)
        .size()
        .rename("candidate_group_size")
        .reset_index()
    )
    candidate_dup = pair_sizes[
        pair_sizes["candidate_group_size"] > 1
    ][["h1", "h2"]]
    dup_index = pd.MultiIndex.from_frame(candidate_dup)

    # Every candidate duplicate group is re-read and directly verified.
    pair_exemplar = {}
    strong_exemplar = {}
    collision_pairs = set()
    collision_row_digest = {}

    if len(candidate_dup):
        read_cols = list(MODEL_COLUMNS)
        for c in FINGERPRINT_METADATA_READ_COLUMNS:
            if c not in read_cols:
                read_cols.append(c)

        verify_offset = 0
        for chunk in iter_master(
            MASTER_DATASET_PATH,
            columns=read_cols,
            chunksize=CHUNK_SIZE,
        ):
            h1, h2 = hash128_frame(
                chunk[MODEL_COLUMNS],
                MODEL_COLUMNS,
                NUMERICAL_MODEL_COLUMNS,
            )
            idx = pd.MultiIndex.from_arrays([h1, h2])
            mask = idx.isin(dup_index)

            for pos in np.flatnonzero(mask):
                pair = (int(h1[pos]), int(h2[pos]))
                raw = canonical_row_bytes(
                    chunk[MODEL_COLUMNS].iloc[int(pos)],
                    MODEL_COLUMNS,
                    set(NUMERICAL_MODEL_COLUMNS),
                )
                if pair not in pair_exemplar:
                    pair_exemplar[pair] = raw
                elif pair_exemplar[pair] != raw:
                    collision_pairs.add(pair)

                digest = hashlib.sha256(raw).hexdigest()
                skey = (pair, digest)
                if skey not in strong_exemplar:
                    strong_exemplar[skey] = raw
                elif strong_exemplar[skey] != raw:
                    record_failure(
                        7,
                        "An unresolved SHA-256 collision was detected during direct "
                        "raw-value duplicate verification."
                    )
                    raise RuntimeError(
                        "Unresolved fingerprint collision; partitioning is unsafe."
                    )
                collision_row_digest[verify_offset + int(pos)] = digest

            verify_offset += len(chunk)
            del chunk, h1, h2, idx
            gc.collect()

    # Candidate fingerprint.
    base["feature_fingerprint"] = [
        format_h128(a, b)
        for a, b in zip(base["h1"].to_numpy(), base["h2"].to_numpy())
    ]

    # Any effective-128-bit collision is split by cryptographic digest.
    if collision_pairs:
        mask = np.array([
            (int(a), int(b)) in collision_pairs
            for a, b in zip(base["h1"].to_numpy(), base["h2"].to_numpy())
        ])
        ordinals = base.loc[mask, "row_ordinal_0based"].to_numpy(dtype=np.int64)
        replacements = []
        for ordinal in ordinals:
            digest = collision_row_digest.get(int(ordinal))
            if digest is None:
                record_failure(
                    7,
                    "A collided candidate group lacks a collision-resolution digest."
                )
                raise RuntimeError("Incomplete fingerprint collision resolution.")
            replacements.append("sha256:" + digest)
        base.loc[mask, "feature_fingerprint"] = replacements

    groups = (
        base.groupby("feature_fingerprint", sort=False)
        .agg(
            fingerprint_group_size=("row_ordinal_0based", "size"),
            binary_label_nunique=("binary_label", "nunique"),
            binary_label_min=("binary_label", "min"),
            binary_label_max=("binary_label", "max"),
            attack_family_first=("attack_family_metadata", "first"),
        )
        .reset_index()
    )

    groups["binary_label_set"] = np.select(
        [
            groups["binary_label_nunique"].eq(1) & groups["binary_label_min"].eq(0),
            groups["binary_label_nunique"].eq(1) & groups["binary_label_min"].eq(1),
            groups["binary_label_nunique"].gt(1),
        ],
        ['[0]', '[1]', '[0,1]'],
        default='UNRESOLVED',
    )

    groups["fingerprint_group_class"] = np.select(
        [
            groups["fingerprint_group_size"].eq(1),
            groups["fingerprint_group_size"].gt(1)
            & groups["binary_label_nunique"].eq(1),
            groups["binary_label_nunique"].gt(1),
        ],
        [
            "unique",
            "same-label duplicate group",
            "mixed-label/conflicting group",
        ],
        default="UNRESOLVED",
    )

    if groups["fingerprint_group_class"].eq("UNRESOLVED").any():
        record_failure(7, "At least one fingerprint group could not be classified.")

    base = base.merge(
        groups[
            ["feature_fingerprint", "fingerprint_group_size",
             "fingerprint_group_class"]
        ],
        on="feature_fingerprint",
        how="left",
        validate="many_to_one",
    )

    # Exact family-set reporting for non-unique groups; unique groups are represented
    # directly by attack_family_first.
    nonunique_fps = set(
        groups.loc[
            groups["fingerprint_group_size"] > 1, "feature_fingerprint"
        ].tolist()
    )
    if nonunique_fps:
        subset = base[
            base["feature_fingerprint"].isin(nonunique_fps)
        ][["feature_fingerprint", "attack_family_metadata"]].copy()

        family_sets = (
            subset.groupby("feature_fingerprint", sort=False)[
                "attack_family_metadata"
            ]
            .agg(
                lambda s: json.dumps(
                    sorted({str(x) for x in s.dropna().tolist()}),
                    ensure_ascii=False,
                )
            )
            .rename("attack_family_set_json")
            .reset_index()
        )
        groups = groups.merge(
            family_sets,
            on="feature_fingerprint",
            how="left",
            validate="one_to_one",
        )
    else:
        groups["attack_family_set_json"] = None

    def unique_family_json(v):
        if pd.isna(v):
            return "[]"
        return json.dumps([str(v)], ensure_ascii=False)

    unique_mask = groups["fingerprint_group_size"].eq(1)
    groups.loc[unique_mask, "attack_family_set_json"] = (
        groups.loc[unique_mask, "attack_family_first"].map(unique_family_json)
    )

    # Row-level mapping written in bounded chunks; no feature matrix is duplicated.
    group_lookup = groups.set_index("feature_fingerprint")[
        ["fingerprint_group_size", "fingerprint_group_class"]
    ]

    def row_mapping_chunks():
        for start in range(0, len(base), ROW_ID_CHUNK_SIZE):
            part = base.iloc[start:start + ROW_ID_CHUNK_SIZE].copy()
            ords = part["row_ordinal_0based"].to_numpy(dtype=np.int64)
            _, _, row_ids = provenance_for_ordinals(
                ords, SOURCE_RANGES, DATASET_PREFIX
            )

            part["row_id"] = row_ids
            part["future_partition_eligibility"] = np.select(
                [
                    part["fingerprint_group_class"].eq(
                        "mixed-label/conflicting group"
                    ),
                    part["binary_label"].eq(1),
                    part["binary_label"].eq(0),
                ],
                [
                    "development_or_final_only_indivisible",
                    "development_or_final_only_indivisible",
                    "benign_partition_eligible_subject_to_group_integrity",
                ],
                default="UNRESOLVED",
            )
            yield part[
                [
                    "row_id",
                    "feature_fingerprint",
                    "fingerprint_group_size",
                    "fingerprint_group_class",
                    "binary_label",
                    "attack_family_metadata",
                    "future_partition_eligibility",
                ]
            ]

    write_chunked_parquet(ROW_GROUP_MAPPING_PATH, row_mapping_chunks())

    duplicate_groups = groups[
        groups["fingerprint_group_class"].eq("same-label duplicate group")
    ].copy()
    conflict_groups = groups[
        groups["fingerprint_group_class"].eq("mixed-label/conflicting group")
    ].copy()

    write_parquet_frozen(DUPLICATE_GROUP_MANIFEST_PATH, duplicate_groups)
    write_parquet_frozen(CONFLICT_GROUP_MANIFEST_PATH, conflict_groups)

    if len(conflict_groups):
        mixed_rows = base[
            base["feature_fingerprint"].isin(
                set(conflict_groups["feature_fingerprint"])
            )
        ].copy()
        mixed_family_summary = (
            mixed_rows.groupby(
                ["feature_fingerprint", "binary_label", "attack_family_metadata"],
                dropna=False,
            )
            .size()
            .rename("support")
            .reset_index()
        )
    else:
        mixed_family_summary = pd.DataFrame(
            columns=[
                "feature_fingerprint", "binary_label",
                "attack_family_metadata", "support"
            ]
        )

    write_csv_frozen(MIXED_FAMILY_SUMMARY_PATH, mixed_family_summary)

    FINGERPRINT_AUDIT = {
        "dataset": DATASET_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "master_dataset_sha256": MASTER_SHA256,
        "created_at_utc": EXECUTION_UTC,
        "algorithm": {
            "implementation_version": "Experiment1_v2_modelview_fp_v1",
            "candidate_fingerprint": (
                "two deterministic pandas 64-bit row hashes with independent fixed "
                "hash keys over the exact ordered frozen raw model-input view"
            ),
            "effective_candidate_bits": 128,
            "candidate_duplicate_verification": (
                "direct canonical raw-value byte equality for every observed candidate "
                "duplicate group; SHA-256 used to split effective-128-bit collisions"
            ),
            "python_builtin_hash_used": False,
            "column_order": MODEL_COLUMNS,
            "numeric_value_rule": (
                "numeric model columns canonicalized as float64; NaN/+Inf/-Inf have "
                "explicit deterministic byte sentinels"
            ),
            "categorical_value_rule": (
                "exact parsed source string with deterministic missing sentinel"
            ),
        },
        "total_observations": int(len(base)),
        "number_of_fingerprint_groups": int(len(groups)),
        "number_of_unique_fingerprint_groups": int(
            groups["fingerprint_group_class"].eq("unique").sum()
        ),
        "number_of_same_label_duplicate_groups": int(len(duplicate_groups)),
        "number_of_observations_in_same_label_duplicate_groups": int(
            duplicate_groups["fingerprint_group_size"].sum()
        ),
        "number_of_mixed_label_fingerprint_groups": int(len(conflict_groups)),
        "number_of_observations_in_mixed_label_groups": int(
            conflict_groups["fingerprint_group_size"].sum()
        ),
        "maximum_fingerprint_group_size": int(
            groups["fingerprint_group_size"].max()
        ),
        "resolved_effective_128bit_collision_groups": int(len(collision_pairs)),
        "all_candidate_duplicate_groups_directly_verified": True,
        "same_label_duplicate_future_rule": (
            "all rows sharing feature_fingerprint remain in one future partition"
        ),
        "mixed_label_future_rule": (
            "remain intact; never enter benign_weight_train, benign_early_stop, or "
            "benign_threshold_calibration; later allocate indivisibly to development "
            "or final test"
        ),
    }
    write_json_frozen(FINGERPRINT_AUDIT_PATH, FINGERPRINT_AUDIT)
    FINGERPRINT_MANIFEST_HASH = sha256_file(ROW_GROUP_MAPPING_PATH)

    if len(duplicate_groups):
        record_warning(
            7,
            f"{len(duplicate_groups):,} same-label model-view duplicate groups were "
            "observed and retained as future indivisible allocation groups."
        )
    if len(conflict_groups):
        record_warning(
            7,
            f"{len(conflict_groups):,} mixed-label/conflicting model-view groups were "
            "observed and retained; they are ineligible for benign-only future partitions."
        )

    if SECTION_STATUS["7"] != "FAIL":
        pass_section(7)

    print("Fingerprint group summary:")
    display(
        groups["fingerprint_group_class"]
        .value_counts()
        .rename_axis("group_class")
        .reset_index(name="group_count")
    )
    print("Mixed-label family support summary:")
    display(mixed_family_summary.head(30))

    del base, groups, duplicate_groups, conflict_groups
    gc.collect()
    BASE_PATH.unlink(missing_ok=True)


[SECTION 7 WARNING] 94,781 same-label model-view duplicate groups were observed and retained as future indivisible allocation groups.
[SECTION 7 WARNING] 698 mixed-label/conflicting model-view groups were observed and retained; they are ineligible for benign-only future partitions.
Fingerprint group summary:


,group_class,group_count
0,unique,2426185
1,same-label duplicate group,94781
2,mixed-label/conflicting group,698


Mixed-label family support summary:


,feature_fingerprint,binary_label,attack_family_metadata,support
0,h128:0018346ab24699060018346ab2469906,0,<NA>,1
1,h128:0018346ab24699060018346ab2469906,1,DoS Hulk,12
2,h128:004fac27a5378ef4004fac27a5378ef4,0,<NA>,1
3,h128:004fac27a5378ef4004fac27a5378ef4,1,PortScan,2
4,h128:005ae93c5b8d82d6005ae93c5b8d82d6,0,<NA>,1
5,h128:005ae93c5b8d82d6005ae93c5b8d82d6,1,DoS Hulk,1
6,h128:00628bd3d315d2b700628bd3d315d2b7,0,<NA>,1
7,h128:00628bd3d315d2b700628bd3d315d2b7,1,PortScan,4
8,h128:009a9b57984511e2009a9b57984511e2,0,<NA>,1
9,h128:009a9b57984511e2009a9b57984511e2,1,DoS Hulk,9


# Artifact inventory


In [12]:
# ============================================================
# 14. ARTIFACT INVENTORY — PREVIEW
# ============================================================
# The final inventory file is written after the readiness reports so those reports
# are included in the traceable artifact list.

artifact_specs = [
    (GLOBAL_PROTOCOL_PATH, "Shared substantive Experiment 1 v2 global protocol", "1"),
    (ENVIRONMENT_MANIFEST_PATH, "Software environment capture", "1"),
    (MASTER_MANIFEST_PATH, "Frozen master provenance manifest", "2"),
    (SCHEMA_AUDIT_PATH, "Post-merge schema audit", "3"),
    (CARDINALITY_AUDIT_PATH, "Exact per-column cardinality audit", "3"),
    (QUALITY_PATH, "Post-merge quality audit", "3"),
    (LABEL_AUDIT_PATH, "Label/taxonomy distribution audit", "3"),
    (NEARCONST_PATH, "Constant/near-constant descriptive audit", "3"),
    (IDENTIFIER_PATH, "Identifier review flags", "3"),
    (SUMMARY_MD_PATH, "Human-readable post-merge audit summary", "3"),
    (TAXONOMY_AUDIT_CSV_PATH, "Taxonomy audit CSV", "4"),
    (TAXONOMY_AUDIT_MD_PATH, "Taxonomy audit Markdown", "4"),
    (TAXONOMY_MANIFEST_PATH, "Frozen taxonomy specification", "4"),
    (ROW_ID_MANIFEST_PATH, "Immutable row-identity specification", "5"),
    (ROW_IDENTITY_PATH, "Row-level source provenance identity", "5"),
    (FEATURE_POLICY_PATH, "Frozen raw feature policy", "6"),
    (FINGERPRINT_AUDIT_PATH, "Model-view fingerprint audit", "7"),
    (ROW_GROUP_MAPPING_PATH, "Row-level model-view group mapping", "7"),
    (DUPLICATE_GROUP_MANIFEST_PATH, "Same-label duplicate-group manifest", "7"),
    (CONFLICT_GROUP_MANIFEST_PATH, "Mixed-label/conflicting-group manifest", "7"),
    (MIXED_FAMILY_SUMMARY_PATH, "Mixed-label family support summary", "7"),
]

ARTIFACT_RECORDS = []
for p, purpose, section in artifact_specs:
    if Path(p).exists():
        register_artifact(p, purpose, section)

display(pd.DataFrame(ARTIFACT_RECORDS))


,artifact_name,artifact_path,artifact_type,file_size,sha256,purpose,section
0,experiment1_v2_global_protocol.json,/content/drive/MyDrive/EXPERIMENT1_V2/global/e...,json,1809,5146784b4661c7ccc6cefc92d2df47ff4dd0477bcf45c4...,Shared substantive Experiment 1 v2 global prot...,1
1,software_environment.json,/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS...,json,398,0c9114f98cfd23e4636db0f61a95ff83eb66697ba93e97...,Software environment capture,1
2,CICIDS2017_master_manifest.json,/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS...,json,12827,12e1611574c793cfad838fcfae913f01d99deb53b8cd71...,Frozen master provenance manifest,2
3,postmerge_schema_audit.csv,/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS...,csv,18820,4177f7e7b338a2d8da1a6b60f5eeac57f31d7fa3a653ea...,Post-merge schema audit,3
4,column_cardinality_audit.csv,/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS...,csv,2962,134042ae9b7b33f89d8f284e7abb953acfd73820a64e8f...,Exact per-column cardinality audit,3
5,postmerge_quality_audit.json,/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS...,json,3019,058ce7ef7621ac1292f40475499f027c824b3fd51d765d...,Post-merge quality audit,3
6,postmerge_label_audit.csv,/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS...,csv,2951,37d01050deb9e44468d6e525aa206e674a2599f2e86706...,Label/taxonomy distribution audit,3
7,constant_nearconstant_audit.csv,/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS...,csv,16129,7b1ab500ebeef98057eed1dd98d3b09d601a69a75023b9...,Constant/near-constant descriptive audit,3
8,identifier_review.csv,/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS...,csv,55,d914841dec144a3c446631656cc10afdd71b9e6785c81b...,Identifier review flags,3
9,postmerge_audit_summary.md,/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS...,md,298,27520ed521202ec2de47f075dab16733d58b3609887e6e...,Human-readable post-merge audit summary,3


# Section-level validation and final PRE-PARTITION readiness gate


In [13]:
# ============================================================
# 15–17. SECTION VALIDATION + FINAL PRE-PARTITION READINESS GATE
# ============================================================
for s in range(1, 8):
    if SECTION_STATUS[str(s)] == "NOT_RUN":
        record_failure(s, "Section did not complete.")

section_table = pd.DataFrame([
    {"section": int(s), "status": SECTION_STATUS[str(s)]}
    for s in range(1, 8)
])
display(section_table)

if BLOCKING_FAILURES or any(v == "FAIL" for v in SECTION_STATUS.values()):
    FINAL_STATUS = "FAIL"
    NEXT_AUTHORIZED_STAGE = "NONE"
else:
    FINAL_STATUS = (
        "PASS_WITH_NONBLOCKING_WARNINGS"
        if NONBLOCKING_WARNINGS
        else "PASS"
    )
    NEXT_AUTHORIZED_STAGE = (
        "Sections 8–11: group-aware partition design and immutable partition construction"
    )

READINESS_REPORT = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "notebook_stage": NOTEBOOK_STAGE,
    "created_at_utc": EXECUTION_UTC,
    "master_dataset_sha256": globals().get("MASTER_SHA256"),
    "global_protocol_hash": globals().get("GLOBAL_PROTOCOL_HASH"),
    "taxonomy_hash": globals().get("TAXONOMY_HASH"),
    "row_identity_specification_hash": globals().get("ROW_ID_SPEC_HASH"),
    "raw_feature_policy_hash": globals().get("FEATURE_POLICY_HASH"),
    "fingerprint_algorithm": (
        FINGERPRINT_AUDIT.get("algorithm")
        if isinstance(globals().get("FINGERPRINT_AUDIT"), dict)
        else None
    ),
    "fingerprint_manifest_hash": globals().get("FINGERPRINT_MANIFEST_HASH"),
    **{
        f"section_{i}_status": SECTION_STATUS[str(i)]
        for i in range(1, 8)
    },
    "blocking_failures": BLOCKING_FAILURES,
    "nonblocking_warnings": NONBLOCKING_WARNINGS,
    "final_status": FINAL_STATUS,
    "next_authorized_stage": NEXT_AUTHORIZED_STAGE,
}

READINESS_JSON_PATH = READINESS_DIR / "prepartition_readiness_report.json"
READINESS_MD_PATH = READINESS_DIR / "prepartition_readiness_report.md"

write_json_frozen(READINESS_JSON_PATH, READINESS_REPORT)

md_lines = [
    f"# PRE-PARTITION Readiness — {DATASET_NAME}",
    "",
    f"- **Master SHA-256:** `{READINESS_REPORT['master_dataset_sha256']}`",
    f"- **Global protocol hash:** `{READINESS_REPORT['global_protocol_hash']}`",
    f"- **Taxonomy hash:** `{READINESS_REPORT['taxonomy_hash']}`",
    f"- **Row identity specification hash:** `{READINESS_REPORT['row_identity_specification_hash']}`",
    f"- **Raw feature-policy hash:** `{READINESS_REPORT['raw_feature_policy_hash']}`",
    f"- **Blocking failures:** {len(BLOCKING_FAILURES)}",
    f"- **Warnings:** {len(NONBLOCKING_WARNINGS)}",
    f"- **PRE-PARTITION READINESS:** **{FINAL_STATUS}**",
    f"- **Next authorized stage:** {NEXT_AUTHORIZED_STAGE}",
    "",
    "## Section status",
    "",
    markdown_table(section_table),
    "",
    "## Blocking failures",
]
if BLOCKING_FAILURES:
    md_lines.extend([
        f"- Section {x['section']}: {x['message']}"
        for x in BLOCKING_FAILURES
    ])
else:
    md_lines.append("- None")

md_lines.extend(["", "## Nonblocking warnings"])
if NONBLOCKING_WARNINGS:
    md_lines.extend([
        f"- Section {x['section']}: {x['message']}"
        for x in NONBLOCKING_WARNINGS
    ])
else:
    md_lines.append("- None")

write_text_frozen(READINESS_MD_PATH, "\n".join(md_lines) + "\n")

# Final artifact inventory includes readiness evidence.
register_artifact(
    READINESS_JSON_PATH, "Machine-readable pre-partition readiness report", "gate"
)
register_artifact(
    READINESS_MD_PATH, "Human-readable pre-partition readiness report", "gate"
)
ARTIFACT_INVENTORY_PATH = OUTPUT_DIR / "artifact_inventory.csv"
write_csv_frozen(
    ARTIFACT_INVENTORY_PATH,
    pd.DataFrame(ARTIFACT_RECORDS).drop_duplicates(
        subset=["artifact_path"], keep="last"
    ),
)

print("=" * 76)
print(f"Dataset: {DATASET_NAME}")
print("Sections evaluated: 1–7")
print(f"Master SHA-256: {globals().get('MASTER_SHA256')}")
print(f"Global protocol hash: {globals().get('GLOBAL_PROTOCOL_HASH')}")
print(f"Feature-policy hash: {globals().get('FEATURE_POLICY_HASH')}")
if isinstance(globals().get("FINGERPRINT_AUDIT"), dict):
    print(
        "Fingerprint groups:",
        FINGERPRINT_AUDIT.get("number_of_fingerprint_groups")
    )
    print(
        "Mixed-label groups:",
        FINGERPRINT_AUDIT.get("number_of_mixed_label_fingerprint_groups")
    )
else:
    print("Fingerprint groups: UNAVAILABLE")
    print("Mixed-label groups: UNAVAILABLE")
print(f"Blocking failures: {len(BLOCKING_FAILURES)}")
print(f"Warnings: {len(NONBLOCKING_WARNINGS)}")
print(f"PRE-PARTITION READINESS: {FINAL_STATUS}")
print(f"Next authorized stage: {NEXT_AUTHORIZED_STAGE}")
print("=" * 76)

if FINAL_STATUS == "FAIL":
    print(
        "\nPartitioning is NOT authorized. Resolve each blocking failure against "
        "the frozen master/provenance evidence and rerun Sections 1–7."
    )


,section,status
0,1,PASS
1,2,PASS
2,3,PASS
3,4,PASS
4,5,PASS
5,6,PASS
6,7,PASS


Dataset: CICIDS2017
Sections evaluated: 1–7
Master SHA-256: b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff60623871155d51cabd3
Global protocol hash: 55713fd82c01168e0a906d282644b1c8dff88632dcb82512c5f62c637e35ff1f
Feature-policy hash: 3cae0d0bb613b1966f2542c2211afadcf4cdc2bcb6ceb9017ad51703c1b9dc32
Fingerprint groups: 2521664
Mixed-label groups: 698
Blocking failures: 0
Warnings: 4
PRE-PARTITION READINESS: PASS_WITH_NONBLOCKING_WARNINGS
Next authorized stage: Sections 8–11: group-aware partition design and immutable partition construction


# Next-stage authorization statement

Sections 8–11 are authorized only when the readiness report is `PASS` or `PASS_WITH_NONBLOCKING_WARNINGS`. A `FAIL` status authorizes no partition construction.
